## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI. Run this command in your terminal:

```bash
az login
```

or

```bash
az login --use-device-code
```

# 🏦 Azure AI Agent with Function Tools - Banking Operations 💼

This notebook demonstrates function tool integration with `Agent` + `FoundryChatClient` for banking use cases, including account management, transaction history, and loan calculations.

## Features Covered:
- Defining function tools for banking operations
- Agent-level tool configuration (tools available for all queries)
- Run-method tool configuration (tools for specific queries)
- Mixed tool usage patterns
- Multiple banking function coordination

### ⚠️ Important Financial Disclaimer ⚠️
> **This notebook demonstrates simulated banking operations for educational purposes. Always use official banking channels for real financial transactions.**

## Prerequisites

Before running this notebook, ensure you have:
- Azure CLI installed and authenticated (`az login --use-device-code`)
- Access to an Microsoft Foundry project with deployed models
- Environment variables set up in `.env` file:
  - `AI_FOUNDRY_PROJECT_ENDPOINT`
  - `AZURE_AI_MODEL_DEPLOYMENT_NAME`

## Import Libraries

Import the required libraries using the `Agent` + `FoundryChatClient` pattern:

In [ ]:
# Copyright (c) Microsoft. All rights reserved.
import asyncio
import os
import sys
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from random import randint, uniform
from typing import Annotated

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from pydantic import Field

assert version("agent-framework-core") == "1.17.0", "Select the pinned project kernel."
repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements.in").is_file())
load_dotenv(repo_root / ".env", override=False)

endpoint = (
    os.getenv("FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    or os.getenv("AZURE_AI_PROJECT_ENDPOINT")
)
model = os.getenv("FOUNDRY_MODEL") or os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
if not endpoint or not model:
    raise ValueError("Set a Foundry project endpoint and model deployment name in the environment.")
assert Path(sys.executable).resolve() == (repo_root / ".venv/Scripts/python.exe").resolve(), (
    "Select the repository .venv kernel."
)
print(f"Kernel: {sys.executable}\nPython: {sys.version.split()[0]}")
print("Project endpoint and model: configured (values hidden)")

## Define Function Tools 🏦

Let's define banking-related functions that our agent can use for account operations:

In [ ]:
def get_account_balance(
    account_id: Annotated[str, Field(description="The customer account ID to check balance for.")],
) -> str:
    """Get the current balance for a customer account."""
    # Simulated account balances
    balance = round(uniform(1000, 50000), 2)
    return f"Account {account_id}: Current balance is ${balance:,.2f} as of {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}"


def get_transaction_history(
    account_id: Annotated[str, Field(description="The account ID to get transaction history for.")],
    num_transactions: Annotated[int, Field(description="Number of recent transactions to retrieve.")] = 5,
) -> str:
    """Get recent transaction history for an account."""
    transactions = []
    categories = ["Deposit", "Withdrawal", "Transfer", "Payment", "Interest"]
    
    for i in range(num_transactions):
        amount = round(uniform(10, 2000), 2)
        category = categories[randint(0, len(categories)-1)]
        sign = "+" if category in ["Deposit", "Interest"] else "-"
        transactions.append(f"  {i+1}. {category}: {sign}${amount:,.2f}")
    
    return f"Recent transactions for account {account_id}:\n" + "\n".join(transactions)


def calculate_loan_payment(
    principal: Annotated[float, Field(description="The loan principal amount in dollars.")],
    annual_rate: Annotated[float, Field(description="The annual interest rate as a percentage (e.g., 6.5 for 6.5%).")],
    term_months: Annotated[int, Field(description="The loan term in months.")],
) -> str:
    """Calculate the monthly payment for a loan."""
    monthly_rate = annual_rate / 100 / 12
    if monthly_rate == 0:
        monthly_payment = principal / term_months
    else:
        monthly_payment = principal * (monthly_rate * (1 + monthly_rate)**term_months) / ((1 + monthly_rate)**term_months - 1)
    
    total_payment = monthly_payment * term_months
    total_interest = total_payment - principal
    
    return f"""Loan Calculation:
  Principal: ${principal:,.2f}
  Annual Rate: {annual_rate}%
  Term: {term_months} months
  Monthly Payment: ${monthly_payment:,.2f}
  Total Interest: ${total_interest:,.2f}
  Total Payment: ${total_payment:,.2f}"""

## Pattern 1: Tools Defined on Agent Level 🔧

In this pattern, tools are provided when creating the agent. The agent can use these banking tools for any query during its lifetime:

In [ ]:
async def tools_on_agent_level() -> None:
    """Use tools configured on the agent for multiple independent runs."""
    print("=== 🏦 Pattern 1: Tools Defined on Agent Level ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="BankingAssistant",
                instructions=(
                    "You are a helpful Banking Assistant that can check account balances, show transaction "
                    "history, and calculate loan payments."
                ),
                tools=[get_account_balance, get_transaction_history, calculate_loan_payment],
            )
            queries = [
                "What's the balance for account ACC-12345?",
                "Show me the last 3 transactions for account ACC-12345",
                "Calculate the monthly payment for a $250,000 mortgage at 6.5% for 30 years",
            ]
            for query in queries:
                print(f"\n🤔 Customer: {query}")
                async with asyncio.timeout(90):
                    result = await agent.run(query)
                assert result.text, "The service returned no answer."
                print(f"🏦 Assistant: {result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


await tools_on_agent_level()

## Pattern 2: Tools Passed to Run Method 🎯

In this pattern, tools are passed to the `run` method, allowing for different banking tools per query:

In [ ]:
async def tools_on_run_level() -> None:
    """Provide a different tool set for each run."""
    print("=== 🎯 Pattern 2: Tools Passed to Run Method ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="BankingAssistant",
                instructions="You are a helpful Banking Assistant. Use the provided tools.",
            )
            requests = [
                ("What's my balance for account ACC-98765?", [get_account_balance]),
                ("Calculate payments for a $50,000 auto loan at 7% for 5 years", [calculate_loan_payment]),
                (
                    "Check balance for ACC-98765 and calculate a personal loan of $10,000 at 10% for 3 years",
                    [get_account_balance, calculate_loan_payment],
                ),
            ]
            for query, tools in requests:
                print(f"\n🤔 Customer: {query}")
                async with asyncio.timeout(90):
                    result = await agent.run(query, tools=tools)
                assert result.text, "The service returned no answer."
                print(f"🏦 Assistant: {result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


await tools_on_run_level()

## Pattern 3: Mixed Tools (Agent + Run Method) 🔄

This pattern combines agent-level tools with additional run-method tools for comprehensive banking services:

In [ ]:
async def mixed_tools_example() -> None:
    """Combine agent-level and run-level tools."""
    print("=== 🔄 Pattern 3: Mixed Tools (Agent + Run Method) ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="BankingAssistant",
                instructions="You are a comprehensive Banking Assistant.",
                tools=[get_account_balance],
            )
            query = (
                "Check balance for ACC-54321, show last 4 transactions, and calculate a $100,000 "
                "home equity loan at 8% for 15 years"
            )
            print(f"\n🤔 Customer: {query}")
            async with asyncio.timeout(90):
                result = await agent.run(query, tools=[get_transaction_history, calculate_loan_payment])
            assert result.text, "The service returned no answer."
            print(f"🏦 Assistant: {result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()


await mixed_tools_example()

## Advanced Function Tools 📊

Let's create more sophisticated banking tools for investment and credit operations:

In [ ]:
def get_credit_score(
    customer_id: Annotated[str, Field(description="The customer ID to check credit score for.")],
) -> str:
    """Get the credit score for a customer."""
    score = randint(580, 850)
    rating = "Excellent" if score >= 750 else "Good" if score >= 700 else "Fair" if score >= 650 else "Poor"
    return f"Customer {customer_id}: Credit Score is {score} ({rating})"


def get_investment_portfolio(
    account_id: Annotated[str, Field(description="The investment account ID.")],
) -> str:
    """Get investment portfolio summary."""
    stocks = round(uniform(10000, 50000), 2)
    bonds = round(uniform(5000, 25000), 2)
    cash = round(uniform(1000, 10000), 2)
    total = stocks + bonds + cash
    
    return f"""Investment Portfolio for {account_id}:
  Stocks: ${stocks:,.2f} ({stocks/total*100:.1f}%)
  Bonds: ${bonds:,.2f} ({bonds/total*100:.1f}%)
  Cash: ${cash:,.2f} ({cash/total*100:.1f}%)
  Total Value: ${total:,.2f}
  Last Updated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}"""


def calculate_compound_interest(
    principal: Annotated[float, Field(description="Initial investment amount.")],
    annual_rate: Annotated[float, Field(description="Annual interest rate as percentage.")],
    years: Annotated[int, Field(description="Number of years.")],
    compounds_per_year: Annotated[int, Field(description="Number of times interest compounds per year (12 for monthly, 4 for quarterly).")] = 12,
) -> str:
    """Calculate compound interest growth."""
    final_amount = principal * (1 + annual_rate/100/compounds_per_year) ** (compounds_per_year * years)
    total_interest = final_amount - principal
    
    return f"""Compound Interest Calculation:
  Initial Investment: ${principal:,.2f}
  Annual Rate: {annual_rate}%
  Term: {years} years
  Compounding: {compounds_per_year}x per year
  Final Value: ${final_amount:,.2f}
  Total Interest Earned: ${total_interest:,.2f}"""

## Comprehensive Financial Advisor Example 💼

Let's test a comprehensive Financial Advisor with all banking tools:

In [ ]:
async def comprehensive_financial_advisor() -> None:
    """Run independent consultations with the complete tool set."""
    print("=== 💼 Comprehensive Financial Advisor ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="FinancialAdvisor",
                instructions=(
                    "You are a comprehensive Financial Advisor. Use the available tools and remind customers "
                    "to consult professionals for major financial decisions."
                ),
                tools=[
                    get_account_balance,
                    get_transaction_history,
                    calculate_loan_payment,
                    get_credit_score,
                    get_investment_portfolio,
                    calculate_compound_interest,
                ],
            )
            queries = [
                "Check my credit score for customer ID CUST-001 and show my investment portfolio for INV-001",
                "If I invest $25,000 at 7% annual interest compounded monthly for 20 years, how much will I have?",
                (
                    "I want to take a $200,000 mortgage at 6.25% for 30 years. What's my monthly payment and "
                    "can you also check my account ACC-001 balance?"
                ),
            ]
            for index, query in enumerate(queries, 1):
                print(f"\n--- 💼 Financial Consultation {index} ---")
                print(f"🤔 Customer: {query}")
                async with asyncio.timeout(90):
                    result = await agent.run(query)
                assert result.text, "The service returned no answer."
                print(f"🏦 Advisor: {result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()

    print("\n⚠️ Disclaimer: All financial information is simulated for demonstration purposes.")


await comprehensive_financial_advisor()

## Full Banking Consultation Example 🏦

A complete banking consultation scenario:

In [ ]:
async def banking_consultation_scenario() -> None:
    """Simulate one complete banking consultation."""
    print("=== 🏦 Complete Banking Consultation Scenario ===")

    with AzureCliCredential() as credential:
        client = FoundryChatClient(project_endpoint=endpoint, model=model, credential=credential)
        try:
            agent = Agent(
                client=client,
                name="BankingConsultant",
                instructions=(
                    "You are a senior Banking Consultant. Use the available tools to review account status, "
                    "credit standing, investments, and loan options."
                ),
                tools=[
                    get_account_balance,
                    get_transaction_history,
                    get_credit_score,
                    get_investment_portfolio,
                    calculate_loan_payment,
                    calculate_compound_interest,
                ],
            )
            review_request = """Please conduct a complete financial review for customer CUST-2024:
1. Check checking account ACC-2024-CHK balance
2. Show last 5 transactions
3. Check credit score
4. Review investment portfolio INV-2024
5. Calculate what a $150,000 home loan at 6.75% for 30 years would cost
6. Project growth of $10,000 invested at 8% for 10 years"""
            print(f"🤔 Review Request: {review_request}\n")
            async with asyncio.timeout(90):
                result = await agent.run(review_request)
            assert result.text, "The service returned no answer."
            print(f"📊 Comprehensive Review:\n{result.text}")
        finally:
            await client.client.close()
            await client.project_client.close()

    print("\n✅ Financial consultation complete!")
    print("⚠️ Note: All data is simulated for demonstration purposes.")


await banking_consultation_scenario()

## Key Takeaways 📚

### API Pattern

```python
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)

agent = Agent(
    client=client,
    name="BankingAssistant",
    instructions="...",
    tools=[tool1, tool2],
)
result = await agent.run("query")
```

### Tool Configuration Patterns

1. **Agent-Level Tools**: Defined at agent creation, available for all queries
2. **Run-Level Tools**: Passed to specific `run()` calls
3. **Mixed Tools**: Combine both patterns for flexible tool access

### Function Tools Demonstrated

| Tool | Purpose |
|------|---------|
| `get_account_balance` | Check account balances |
| `get_transaction_history` | View recent transactions |
| `calculate_loan_payment` | Calculate loan payments |
| `get_credit_score` | Check credit scores |
| `get_investment_portfolio` | Review investment holdings |
| `calculate_compound_interest` | Project investment growth |

### Best Practices

1. **Tool Typing**: Use `Annotated` and `Field` for clear parameter descriptions
2. **Docstrings**: Include helpful docstrings for tool functions
3. **Error Handling**: Handle missing or invalid inputs gracefully
4. **Disclaimers**: Always include appropriate financial disclaimers

### Use Cases

- **Account Management**: Balance checks, transaction history
- **Loan Services**: Payment calculations, rate comparisons
- **Investment Services**: Portfolio review, growth projections
- **Credit Services**: Credit score checks, eligibility assessments